# Macro Repricing Asset Edge

## tl;dr

- The true local FedWatch track currently contains only six thinned timestamps from one day. It is a **case study**, not a statistical edge estimate.
- The new five-minute release panel contains 14 scheduled anchors. At a five-minute observation delay, six independent events crossed a dovish proxy threshold; their next 60-minute SOXX mean was about **-1.02% gross / -1.07% after 5 bp**, while SOXL was about **-2.93% after 5 bp**. This is a small-sample counterexample to “dovish = buy semiconductors,” not a confirmed short edge.
- The event split suggests a distinction between policy relief and growth-scare dovishness: the one FOMC case rose, while several NFP/CPI/PPI/PCE cases fell. Event counts are too small for inference.
- The long daily proxy track still finds a separate candidate: US2Y-led dovish repricing is followed by positive 5-session average returns in QQQ and SOXX. Horizon and shock type therefore matter.
- No strategy or trade recommendation is promoted from this iteration.

## Context & Methods

This companion notebook reads deterministic outputs created by `run_research.py`. The true-monitor, scheduled five-minute proxy, and long-history daily proxy tracks remain separate.

### Key Assumptions

- Signal timestamps are UTC internally; Beijing time is presentation-only.
- Scheduled-release classification observes the first 5, 10, or 15 minutes, then enters strictly after that decision time plus a 0, 5, or 10 minute execution delay.
- `ZQV26.CBT` is an October 2026 Fed Funds Futures proxy and `ZT=F` is a 2Y Treasury futures price diagnostic. Neither is historical FedWatch probability or an exact 2Y yield.
- CNBC/Tradeweb provides exact yields only for the short recent intraday window.
- Daily uncertainty is resampled by natural-week cluster; scheduled intraday uncertainty is clustered by event id.
- Reported p-values are Benjamini-Hochberg adjusted across each declared test family.

## Data

### 1. Load generated evidence

In [1]:
from pathlib import Path
import json
import pandas as pd

root = Path.cwd()
if not (root / 'research' / 'macro_repricing_asset_edge').exists():
    candidates = [Path.cwd(), *Path.cwd().parents]
    root = next(path for path in candidates if (path / 'research' / 'macro_repricing_asset_edge').exists())
output = root / 'research' / 'macro_repricing_asset_edge' / 'outputs'

meta = json.loads((output / 'run_meta.json').read_text(encoding='utf-8'))
inventory = json.loads((output / 'data_inventory.json').read_text(encoding='utf-8'))
intraday_events = pd.read_csv(output / '02_intraday_events.csv')
intraday_summary = pd.read_csv(output / '04_intraday_edge_summary.csv')
net = pd.read_csv(output / '09_daily_proxy_net_edge_summary.csv')
folds = pd.read_csv(output / '10_daily_proxy_fold_summary.csv')
ablation = pd.read_csv(output / '11_daily_proxy_ablation_summary.csv')
static = pd.read_csv(output / '13_static_background_summary.csv')
scheduled_signals = pd.read_csv(output / '15_scheduled_proxy_signals.csv')
scheduled_cost = pd.read_csv(output / '19_scheduled_latency_cost_summary.csv')
scheduled_loo = pd.read_csv(output / '20_scheduled_leave_one_event_out.csv')
scheduled_quality = pd.read_csv(output / '22_scheduled_data_quality.csv')
meta['counts']

{'monitor_observations': 53524,
 'monitor_panel_rows': 10185,
 'intraday_events': 6,
 'intraday_aligned_rows': 120,
 'daily_proxy_events': 3630,
 'daily_proxy_aligned_rows': 7152,
 'static_background_events': 1193,
 'scheduled_release_anchors': 14,
 'scheduled_proxy_signals': 42,
 'scheduled_proxy_non_stable_signals': 20,
 'scheduled_asset_response_rows': 1112}

### 2. Inspect source coverage

In [2]:
coverage = pd.DataFrame(inventory['datasets'])
coverage[['source', 'dataset', 'rows', 'start_utc', 'end_utc', 'notes']]

,source,dataset,rows,start_utc,end_utc,notes
0,local SQLite,true FedWatch / US2Y / US10Y / DXY observations,53524,2026-08-28 09:30:00.743000+00:00,2026-08-29 15:34:13.242000+00:00,true live-monitor data; short local history
1,Alpaca SIP,SOXX 1Min split-adjusted,620,2026-08-28 08:00:00+00:00,2026-08-28 23:59:00+00:00,listed ETF; true-event window
2,Alpaca SIP,SOXL 1Min split-adjusted,945,2026-08-28 08:00:00+00:00,2026-08-28 23:59:00+00:00,listed ETF; true-event window
3,Alpaca SIP,QQQ 1Min split-adjusted,866,2026-08-28 08:00:00+00:00,2026-08-28 23:59:00+00:00,listed ETF; true-event window
4,OKX local cache,ETH-USDT-SWAP 1D,1719,2021-10-01 16:00:00+00:00,2026-06-15 16:00:00+00:00,
5,OKX public market data,ETH-USDT-SWAP 1m research cache,1946,2026-08-28 05:48:00+00:00,2026-08-29 14:13:00+00:00,perpetual swap
6,OKX public market data,XAU-USDT-SWAP 1m research cache,1946,2026-08-28 05:48:00+00:00,2026-08-29 14:13:00+00:00,perpetual swap; not the listed ETF
7,FRED,DGS2 / DGS10 daily,16868,1962-01-02 00:00:00,2026-08-27 00:00:00,daily background
8,Yahoo Finance,SOXX daily,1924,2019-01-02 14:30:00+00:00,2026-08-27 13:30:00+00:00,listed ETF; adjusted daily history
9,Yahoo Finance,SOXL daily,1924,2019-01-02 14:30:00+00:00,2026-08-27 13:30:00+00:00,listed 3x ETF; measured directly


## Results

### 3. Review true-monitor events

In [3]:
intraday_events[['timestamp_utc', 'regime', 'severity', 'score', 'drivers']]

,timestamp_utc,regime,severity,score,drivers
0,2026-08-28 14:03:51.692000+00:00,hawkish,1,1.366667,US2Y 5m=+4.10/3.00
1,2026-08-28 14:05:36.763000+00:00,hawkish,2,2.100000,US10Y 5m=+3.20/3.00; US2Y 5m=+3.10/3.00
2,2026-08-28 14:11:42.754000+00:00,hawkish,3,2.280000,FedWatch Bias 15m=-6.00/5.00; US2Y 15m=+5.40/5.00
3,2026-08-28 14:41:52.930000+00:00,hawkish,1,1.000000,FedWatch Bias 60m=-10.00/10.00
4,2026-08-28 14:42:23.357000+00:00,hawkish,3,7.328750,FedWatch Bias 15m=-11.70/5.00; FedWatch Bias 6...
5,2026-08-28 15:12:25.638000+00:00,hawkish,1,1.400000,FedWatch Bias 60m=-14.00/10.00


The six rows are all from one event day and therefore share one independent day cluster. Their asset responses are useful for timing forensics, not inference.

In [4]:
intraday_summary[['asset', 'regime', 'horizon_minutes', 'events', 'independent_clusters',
                  'mean_return_pct', 'mean_mae_pct', 'mean_mfe_pct', 'evidence_grade']]

,asset,regime,horizon_minutes,events,independent_clusters,mean_return_pct,mean_mae_pct,mean_mfe_pct,evidence_grade
0,ETH-USDT-SWAP,hawkish,5,6,1,-0.141065,-0.358371,0.138943,case-study only
1,ETH-USDT-SWAP,hawkish,15,6,1,0.018804,-0.577227,0.278903,case-study only
2,ETH-USDT-SWAP,hawkish,30,6,1,0.197849,-0.674464,0.713789,case-study only
3,ETH-USDT-SWAP,hawkish,60,6,1,-0.261173,-1.129293,0.969900,case-study only
4,QQQ,hawkish,5,6,1,0.033210,-0.092608,0.121207,case-study only
5,QQQ,hawkish,15,6,1,0.176849,-0.147522,0.241937,case-study only
6,QQQ,hawkish,30,6,1,0.201843,-0.189497,0.333011,case-study only
7,QQQ,hawkish,60,6,1,0.197605,-0.313869,0.518383,case-study only
8,SOXL,hawkish,5,6,1,-0.178979,-0.607630,0.453715,case-study only
9,SOXL,hawkish,15,6,1,0.063844,-1.108117,0.651899,case-study only


### 4. Test scheduled releases at five-minute resolution

Stable rows remain visible so threshold coverage can be audited rather than silently discarded.

In [5]:
scheduled_signals.groupby(['signal_delay_minutes', 'regime'])['event_id'].nunique().unstack(fill_value=0)

regime,proxy_dovish,proxy_hawkish,stable
signal_delay_minutes,,,
5,6,0,8
10,6,1,7
15,6,1,7


In [6]:
scheduled_60m = scheduled_cost.query(
    "signal_delay_minutes == 5 and execution_delay_minutes == 0 and cost_stress_bp == 5 and horizon_minutes == 60"
)
scheduled_60m[['asset', 'regime', 'events', 'independent_clusters', 'mean_return_pct',
               'bootstrap_95_low_pct', 'bootstrap_95_high_pct',
               'bh_adjusted_p_value', 'evidence_grade']].sort_values('mean_return_pct')

,asset,regime,events,independent_clusters,mean_return_pct,bootstrap_95_low_pct,bootstrap_95_high_pct,bh_adjusted_p_value,evidence_grade
308,SOXL,proxy_dovish,6,6,-2.932981,-5.628262,-0.639599,0.280229,case-study only
312,SOXX,proxy_dovish,6,6,-1.067242,-1.982124,-0.242286,0.280229,case-study only
304,QQQ,proxy_dovish,6,6,-0.366730,-0.592961,-0.141529,0.280229,case-study only
316,XAU-USDT-SWAP,proxy_dovish,5,5,0.021733,-0.102561,0.221938,0.837776,case-study only
300,ETH-USDT-SWAP,proxy_dovish,5,5,1.506151,0.274148,2.705657,0.287338,case-study only


The negative SOXX/SOXL response survives deletion of any one of the six events, but six events are still far below a decision-grade sample. The result is best treated as a warning that dovish repricing can reflect bad growth news.

In [7]:
scheduled_loo.query(
    "signal_delay_minutes == 5 and execution_delay_minutes == 0 and horizon_minutes == 60"
)[['asset', 'regime', 'events', 'full_mean_pct', 'loo_min_mean_pct',
   'loo_max_mean_pct', 'same_sign_all_loo']]

,asset,regime,events,full_mean_pct,loo_min_mean_pct,loo_max_mean_pct,same_sign_all_loo
3,ETH-USDT-SWAP,proxy_dovish,5,1.506151,1.165112,2.017788,True
39,QQQ,proxy_dovish,6,-0.366730,-0.450650,-0.282500,True
75,SOXL,proxy_dovish,6,-2.932981,-3.811574,-1.778353,True
108,SOXX,proxy_dovish,6,-1.067242,-1.381958,-0.661904,True
144,XAU-USDT-SWAP,proxy_dovish,5,0.021733,-0.076086,0.057776,False


### 5. Audit five-minute source quality

In [8]:
scheduled_quality[['dataset', 'rows', 'start_utc', 'end_utc', 'duplicate_timestamps',
                   'null_close_pct', 'median_interval_minutes',
                   'expected_interval_share_pct', 'status']]

,dataset,rows,start_utc,end_utc,duplicate_timestamps,null_close_pct,median_interval_minutes,expected_interval_share_pct,status
0,macro:ZQ_POST_FOMC,6623,2026-06-18 04:15:00+00:00,2026-08-28 20:55:00+00:00,0,0.0,5.0,67.321051,ok
1,macro:ZT,13703,2026-06-18 04:00:00+00:00,2026-08-28 21:00:00+00:00,0,0.0,5.0,98.722814,ok
2,macro:US10Y_YAHOO,4000,2026-06-18 12:20:00+00:00,2026-08-28 18:55:00+00:00,0,0.0,5.0,98.774694,ok
3,macro:DXY,13550,2026-06-18 04:00:00+00:00,2026-08-28 20:55:00+00:00,0,0.0,5.0,99.527640,ok
4,macro:US2Y_EXACT,1746,2026-08-19 15:25:00+00:00,2026-08-28 21:00:00+00:00,0,0.0,5.0,94.899713,ok
5,macro:US10Y_EXACT,1846,2026-08-19 15:26:00+00:00,2026-08-28 20:56:00+00:00,0,0.0,5.0,97.723577,ok
6,asset:SOXX,11508,2026-06-04 08:00:00+00:00,2026-08-28 23:59:53+00:00,0,0.0,5.0,99.382984,ok
7,asset:SOXL,11342,2026-06-04 08:00:00+00:00,2026-08-28 23:59:56+00:00,0,0.0,5.0,98.941892,ok
8,asset:QQQ,11521,2026-06-04 08:00:00+00:00,2026-08-28 23:59:57+00:00,0,0.0,5.0,99.479167,ok
9,asset:ETH-USDT-SWAP,18720,2026-06-25 15:30:00+00:00,2026-08-29 15:25:00+00:00,0,0.0,5.0,100.000000,ok


### 6. Rank long-history proxy results after cost

In [9]:
columns = ['asset', 'regime', 'horizon_sessions', 'events', 'independent_clusters',
           'mean_return_pct', 'bootstrap_95_low_pct', 'bootstrap_95_high_pct',
           'bh_adjusted_p_value', 'evidence_grade']
net.sort_values(['bh_adjusted_p_value', 'events'], ascending=[True, False])[columns].head(15)

,asset,regime,horizon_sessions,events,independent_clusters,mean_return_pct,bootstrap_95_low_pct,bootstrap_95_high_pct,bh_adjusted_p_value,evidence_grade
14,QQQ,dovish,5,237,167,0.525968,0.250259,1.148825,0.053717,preliminary
26,SOXX,dovish,5,237,167,0.736637,0.363417,1.671408,0.055572,preliminary
20,SOXL,dovish,5,237,167,1.660314,0.692754,4.648025,0.140996,preliminary
13,QQQ,dovish,3,237,167,0.306502,0.046376,0.730190,0.188502,preliminary
28,SOXX,hawkish,3,257,179,0.305570,-0.027302,0.915263,0.255977,preliminary
29,SOXX,hawkish,5,257,179,0.375797,-0.060934,1.152303,0.255977,preliminary
19,SOXL,dovish,3,237,167,1.024173,-0.127799,2.961254,0.255977,preliminary
25,SOXX,dovish,3,237,167,0.444625,0.026782,1.025569,0.255977,preliminary
4,ETH-USDT-SWAP,hawkish,3,222,152,-1.011732,-1.598915,0.057160,0.255977,preliminary
2,ETH-USDT-SWAP,dovish,5,184,130,0.938671,-0.110980,2.492986,0.255977,preliminary


### 7. Verify the candidate is US2Y-led, not FedWatch history

The ablation below prevents an outcome driven mostly by US2Y daily changes from being described as a FedWatch edge.

In [10]:
ablation.sort_values(['signal_subset', 'bh_adjusted_p_value'])[[
    'signal_subset', 'asset', 'regime', 'horizon_sessions', 'events',
    'mean_return_pct', 'bootstrap_95_low_pct', 'bootstrap_95_high_pct',
    'bh_adjusted_p_value'
]].groupby('signal_subset', group_keys=False).head(5)

,signal_subset,asset,regime,horizon_sessions,events,mean_return_pct,bootstrap_95_low_pct,bootstrap_95_high_pct,bh_adjusted_p_value
14,us2y,QQQ,dovish,5,210,0.664203,0.313916,1.174458,0.022445
26,us2y,SOXX,dovish,5,210,0.929304,0.394877,1.773242,0.032512
20,us2y,SOXL,dovish,5,210,2.409673,0.833378,4.764639,0.068087
4,us2y,ETH-USDT-SWAP,hawkish,3,197,-1.049442,-1.773285,-0.059550,0.250842
13,us2y,QQQ,dovish,3,210,0.294351,0.021665,0.658944,0.250842
31,us2y_and_zq_confirmed,ETH-USDT-SWAP,dovish,3,6,5.030449,-0.997035,13.057854,0.831606
32,us2y_and_zq_confirmed,ETH-USDT-SWAP,dovish,5,6,3.927786,0.838158,8.876645,0.831606
33,us2y_and_zq_confirmed,ETH-USDT-SWAP,hawkish,1,13,-1.327530,-3.376022,1.049127,0.831606
36,us2y_and_zq_confirmed,GC=F proxy,dovish,1,15,0.480368,-0.235032,1.164944,0.831606
37,us2y_and_zq_confirmed,GC=F proxy,dovish,3,15,0.739140,-0.590033,1.910739,0.831606


### 8. Check chronological stability

In [11]:
candidate = folds.query("regime == 'dovish' and horizon_sessions == 5 and asset in ['QQQ', 'SOXX', 'SOXL']")
candidate[['fold', 'asset', 'events', 'mean_return_pct', 'bootstrap_95_low_pct',
           'bootstrap_95_high_pct', 'bh_adjusted_p_value']].sort_values(['asset', 'fold'])

,fold,asset,events,mean_return_pct,bootstrap_95_low_pct,bootstrap_95_high_pct,bh_adjusted_p_value
14,test_2025_2026,QQQ,51,0.359243,-0.131628,1.340312,0.507281
44,train_2019_2022,QQQ,87,0.178754,-0.336526,1.515175,0.588801
74,validation_2023_2024,QQQ,99,0.916984,0.335273,1.397113,0.046051
20,test_2025_2026,SOXL,51,2.547720,-0.869412,8.821095,0.507281
50,train_2019_2022,SOXL,87,0.497670,-0.966068,6.436752,0.580738
80,validation_2023_2024,SOXL,99,2.224882,-0.932898,4.290060,0.377682
26,test_2025_2026,SOXX,51,0.999351,-0.028085,3.034299,0.507281
56,train_2019_2022,SOXX,87,0.499246,-0.162309,2.279020,0.540437
86,validation_2023_2024,SOXX,99,0.809917,-0.237682,1.481394,0.354103


### 9. Check static-background evidence

In [12]:
static.sort_values('bh_adjusted_p_value')[columns].head(12)

,asset,regime,horizon_sessions,events,independent_clusters,mean_return_pct,bootstrap_95_low_pct,bootstrap_95_high_pct,bh_adjusted_p_value,evidence_grade
6,GC=F proxy,dovish,1,611,152,-0.093339,-0.260185,-0.057282,0.123612,preliminary
14,QQQ,dovish,5,611,152,0.446859,0.135435,0.794907,0.123612,preliminary
26,SOXX,dovish,5,611,152,0.657606,0.135882,1.242872,0.139580,preliminary
20,SOXL,dovish,5,611,152,1.743802,0.255278,3.676394,0.235129,preliminary
22,SOXL,hawkish,3,580,134,0.456610,-0.408971,2.464001,0.438421,preliminary
23,SOXL,hawkish,5,580,134,0.812905,-0.306689,3.953887,0.438421,preliminary
13,QQQ,dovish,3,611,152,0.195434,-0.054415,0.432280,0.438421,preliminary
28,SOXX,hawkish,3,580,134,0.162244,-0.098959,0.817576,0.438421,preliminary
29,SOXX,hawkish,5,580,134,0.338327,-0.091691,1.304503,0.438421,preliminary
25,SOXX,dovish,3,611,152,0.300890,-0.104802,0.684561,0.438421,preliminary


## Takeaways

1. Preserve the live collector: the main bottleneck for actual FedWatch evidence is independent event-day sample size, not parser availability.
2. Do not map `dovish` directly to `SOXX up`. Split policy relief from growth-scare repricing before testing any directional rule.
3. Keep `US2Y dovish daily change → 5-session SOXX/QQQ response` as a **predeclared long-horizon candidate**, separate from the new intraday counterexample.
4. Re-estimate only after materially more independent release and speech days are collected, and keep delay, costs, event type, leave-one-out and multiple-testing correction fixed.
5. Treat SOXL separately because its leverage path produces much wider uncertainty even when the sign matches SOXX.